In [ ]:
import os  
from dotenv import load_dotenv

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

os.environ['LANGSMITH_ENDPOINT'] = LANGSMITH_ENDPOINT

os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY

os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT

In [7]:
from langchain_groq import ChatGroq
llm=ChatGroq(model_name="llama3-70b-8192")
llm.invoke("hi").content

"Hi! It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [8]:
message=[{"role":"system","content":"you are a helpful assistant"},
         {"role":"user","content":"hi how are you?"}]

In [9]:
result = llm.invoke(message)
print(result.content)

Hi! I'm doing great, thanks for asking! I'm a helpful assistant, so I'm always ready to lend a hand or answer any questions you might have. How about you? How's your day going so far?


In [10]:
class Chatbot:
    def __init__(self,system=""):
        self.system=system
        self.message=[]
        if self.system:
            self.message.append({"role":"system","content":system})
    def __call__(self,message):
        self.message.append({"role":"user","content":message})
        result=self.execute()
        self.message.append({"role":"assistant","content":result})
        return result
        
    def execute(self):
        llm = ChatGroq(model_name="Gemma2-9b-It")
        result = llm.invoke(self.message)
        return result.content

In [11]:
bot=Chatbot(system="you are helpful assistant")

In [12]:

bot("hi how are you?")

"I'm doing well, thank you! As an AI, I don't have feelings, but I'm ready to assist you with any questions or tasks you may have. 😊\n\nHow can I help you today?\n"

In [13]:
bot.message

[{'role': 'system', 'content': 'you are helpful assistant'},
 {'role': 'user', 'content': 'hi how are you?'},
 {'role': 'assistant',
  'content': "I'm doing well, thank you! As an AI, I don't have feelings, but I'm ready to assist you with any questions or tasks you may have. 😊\n\nHow can I help you today?\n"}]

In [14]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop your output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.


Your available actions are:
calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point
syntax if necessary

wikipedia:
e.g. wikipedia: Django
Returns a summary from searching Wikipedia

simon_blog_search:
e.g. simon_blog_search: Python
Search Simon's blog for that term

Example session:
Question: What is the capital of France?
Thought: I should look up France on Wikipedia
Action: wikipedia: France
PAUSE

You will be called again with this:
Observation: France is a country. The capital is Paris.

You then output:
Answer: The capital of France is Paris

Please Note: if you get basic conversation questions like "hi","hello","how are you?",\n
you have to answer "hi","hello","i am good".
""".strip()

In [15]:
prompt = """You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop, you output an Answer related to fitness, health, or exercise.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 1800 * 0.2
Runs a calculation (e.g., calorie intake, BMI, workout progress) and returns the result.

fitness_wiki:
e.g. fitness_wiki: Benefits of Cardio
Returns a summary of fitness-related information from a fitness knowledge base (Wikipedia or similar).

exercise_plan:
e.g. exercise_plan: Beginner full-body workout
Returns a recommended workout plan based on the type of exercise or user level.

nutrition_info:
e.g. nutrition_info: Protein sources for vegetarians
Looks up nutritional information or suggestions for diets.

Example session:
Question: What are the benefits of cardio exercises?
Thought: I should look up information about the benefits of cardio exercises.
Action: fitness_wiki: Benefits of Cardio
PAUSE

You will be called again with this:

Observation: Cardio exercises improve cardiovascular health, increase lung capacity, and aid in weight loss.

You then output:

Answer: Cardio exercises improve cardiovascular health, increase lung capacity, and help with weight loss.
""".strip()

Pattern Breakdown:

^: This matches the start of a string. It means the string must begin with what follows.
Action:: This is a literal match. It means the string must have the text "Action:" at the beginning.
(\w+):
The parentheses () define a capture group. This allows you to extract part of the string that matches this section.
\w+ matches one or more word characters (letters, digits, and underscores). This will capture a word that follows "Action:".
:: This is a literal colon that separates the word matched by (\w+) from the rest of the string.
(.*):
This is another capture group, where .* matches any character (.) zero or more times (*), which means it captures everything that comes after the second colon.
What does it do?
This regex is looking for a string that:

Starts with the text "Action:".
Has a word right after it, separated by a colon.
Then, after another colon, it captures everything that follow

In [16]:
import re
action_re = re.compile('^Action: (\w+): (.*)')

# Sample strings
text1 = "Action: Move: North"
text2 = "Action: Jump: High"
text3 = "Error: Not an Action"

In [17]:
# Testing the pattern
match1 = action_re.match(text1)

In [18]:
match1.group(1)

'Move'

In [19]:
match1.group(2)

'North'

In [20]:
match3 = action_re.match(text3)

In [21]:
match3

In [22]:
import httpx
def wikipedia(query):
    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    })
    return response.json()["query"]["search"][0]["snippet"]

In [23]:
import httpx
def simon_blog_search(query):
    response = httpx.get("https://datasette.simonwillison.net/simonwillisonblog.json", params={
        "sql": """
        select
          blog_entry.title || ': ' || substr(html_strip_tags(blog_entry.body), 0, 1000) as text,
          blog_entry.created
        from
          blog_entry join blog_entry_fts on blog_entry.rowid = blog_entry_fts.rowid
        where
          blog_entry_fts match escape_fts(:q)
        order by
          blog_entry_fts.rank
        limit
          1
        """.strip(),
        "_shape": "array",
        "q": query,
    })
    return response.json()[0]["text"]

In [24]:

def calculate(number):
    return eval(number)

In [25]:
calculate("2+2")

4

In [26]:
calculate("20+2*5")

30

In [27]:
known_actions = {
    "wikipedia": wikipedia,
    "calculate": calculate,
    "simon_blog_search": simon_blog_search
}

In [28]:
def query(question,max_turns=5):
    i = 0
    bot = Chatbot(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception(f"Unknown action: {action}: {action_input}")
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = f"Observation: {observation}"
        else:
            return result
    

In [29]:

prompt

'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop, you output an Answer related to fitness, health, or exercise.\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 1800 * 0.2\nRuns a calculation (e.g., calorie intake, BMI, workout progress) and returns the result.\n\nfitness_wiki:\ne.g. fitness_wiki: Benefits of Cardio\nReturns a summary of fitness-related information from a fitness knowledge base (Wikipedia or similar).\n\nexercise_plan:\ne.g. exercise_plan: Beginner full-body workout\nReturns a recommended workout plan based on the type of exercise or user level.\n\nnutrition_info:\ne.g. nutrition_info: Protein sources for vegetarians\nLooks up nutritional information or suggestions for diets.\n\nExample session:\nQuestion: What ar

In [30]:
bot = Chatbot(prompt)

In [31]:
bot("hi how are you?")

"Okay, I understand. I'm ready to answer your fitness, health, or exercise questions!  Ask away.  \n\n"

In [32]:
next_prompt="Tell me about quantum computing from Wikipedia."

In [33]:
next_prompt="Tell me about the current gdp of india from Wikipedia."
result=bot(next_prompt)

In [34]:
result

'Thought: I need to look up the current GDP of India on Wikipedia.\n\nAction: fitness_wiki: GDP of India \n\nPAUSE \n'

In [35]:
actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]

In [36]:
actions

[<re.Match object; span=(0, 35), match='Action: fitness_wiki: GDP of India '>]

In [37]:

action, action_input = actions[0].groups()

In [38]:
action

'fitness_wiki'

In [39]:
action_input

'GDP of India '

In [40]:
query("hi")

Okay, I understand the loop and my actions. Let's begin! 

**Question:** What is a good beginner full-body workout routine? 


**Thought:** I should recommend a beginner full-body workout routine.

**Action:** exercise_plan: Beginner full-body workout 

**PAUSE**  



"Okay, I understand the loop and my actions. Let's begin! \n\n**Question:** What is a good beginner full-body workout routine? \n\n\n**Thought:** I should recommend a beginner full-body workout routine.\n\n**Action:** exercise_plan: Beginner full-body workout \n\n**PAUSE**  \n"

In [42]:
query("Has Simon written about AI?")

Thought:  I need to figure out if someone named Simon has written about AI.  

Action: calculate: 0  
PAUSE 

 -- running calculate 0  
Observation: 0
Answer: I need more information to determine if someone named Simon has written about AI. Can you please provide me with some context or details about Simon? For example, their profession, area of expertise, or any specific publications you are interested in. 





'Answer: I need more information to determine if someone named Simon has written about AI. Can you please provide me with some context or details about Simon? For example, their profession, area of expertise, or any specific publications you are interested in. \n\n\n'